# Cointegration-consistent state space (v0.5)

Until v0.4 the project ran two separate stories about WTI/Brent:

- a **VECM** that did the forecasting work, using an error-correction term built
  from raw observed log prices;
- a generic **one-factor dynamic factor model** that sat beside it as an
  interpretability aside, with loadings that happened to be nearly equal.

This notebook replaces the second with the common-trends representation that
*is* the cointegration result, so the two agree by construction:

$$
\begin{aligned}
\tau_t &= \tau_{t-1} + \mu + \eta_t \\
s_t &= \phi s_{t-1} + \nu_t, \qquad |\phi| < 1 \\
\log \text{WTI}_t &= \tau_t + 0.5\, s_t + \varepsilon_{1t} \\
\log \text{Brent}_t &= a\, \tau_t - 0.5\, s_t + \varepsilon_{2t}
\end{aligned}
$$

One integrated common trend, one stationary relative component. The implied
cointegrating vector is $\beta = [a, -1]'$; with $a = 1$ it is exactly
$[1, -1]'$ and $s_t$ is the latent WTI-Brent log spread with measurement noise
stripped out.

**Why it should forecast better than the VECM.** The error-correction term is a
contemporaneous function of observed prices, so it inherits every roll artefact
and stale quote in the front-month series. The Kalman filter instead delivers
$E[s_t \mid y_{1:t}]$ — the same dislocation, filtered. On the simulated panel
in `reports/synthetic_demo` this is worth roughly 3 percentage points of CRPS
over the VECM on the spread at every horizon, and it makes the improvement
significant at 1 and 4 weeks where the VECM's was not.

## 1. Setup and data

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from oil_futures_regime import (
    build_macro_features, common_uniforms, cts_state_summary, estimate_spread_half_life,
    filtered_states, fit_cts_model, fit_vecm_scenario_model, load_or_download,
    prepare_residual_pool, simulate, var_johansen_summary,
)

START, END, ANCHOR = "2013-01-01", "2025-03-08", "W-FRI"
CACHE = ROOT / "data" / "cache"

oil = load_or_download("oil", {"WTI": "CL=F", "BRENT": "BZ=F"},
                       START, END, ANCHOR, cache_dir=CACHE, dropna="any")
logp = np.log(oil[["WTI", "BRENT"]]).dropna()
print(f"{len(logp)} weeks, {logp.index.min().date()} -> {logp.index.max().date()}")

## 2. Fit the state space

In [ ]:
cts = fit_cts_model(logp, restrict_trend=True)
summary = cts_state_summary(cts)

display(pd.Series(cts.params).round(6).to_frame("estimate"))
print("\nInterpretable summary")
for k, v in summary.items():
    print(f"  {k:28s} {np.round(v, 5)}")

### Is the symmetric restriction defensible?

Refitting with a free trend loading on Brent gives a direct test: if $a$ is far
from 1, the common trend does not enter the two series one-for-one and the
$[1,-1]$ spread is the wrong object to mean-revert.

In [ ]:
cts_free = fit_cts_model(logp, restrict_trend=False)
a_hat = cts_free.params["trend_loading_brent"]
print(f"free trend loading on Brent: a = {a_hat:.4f}")
print(f"implied cointegrating vector: [{a_hat:.4f}, -1]")
print(f"\nlog-likelihood  restricted {cts.params['loglike']:.2f}"
      f"   free {cts_free.params['loglike']:.2f}")
lr = 2 * (cts_free.params["loglike"] - cts.params["loglike"])
from scipy.stats import chi2
print(f"LR statistic {lr:.3f}, p = {1 - chi2.cdf(max(lr, 0), df=1):.4f}"
      "   (H0: a = 1, i.e. beta = [1, -1])")

## 3. Agreement with Johansen and the VECM

In [ ]:
joh = var_johansen_summary(logp, var_lags=1)
vecm = fit_vecm_scenario_model(logp, coint_rank=1, k_ar_diff=1, deterministic="ci")

beta_joh = np.asarray(vecm.result.beta).ravel()
beta_joh = beta_joh / beta_joh[0]                     # normalize on WTI
beta_cts = summary["beta_implied"] / summary["beta_implied"][0]

raw_spread = (logp["WTI"] - logp["BRENT"]).rename("spread")
hl_direct = estimate_spread_half_life(raw_spread)

comparison = pd.DataFrame({
    "Johansen trace stat": [joh["johansen_trace"][0], joh["johansen_trace"][1]],
    "95% critical": [joh["johansen_crit_95"][0], joh["johansen_crit_95"][1]],
}, index=["r = 0", "r <= 1"])
display(comparison.round(3))
print(f"Johansen implied rank at 95%: {joh['rank95']}")

print("\nCointegrating vector (normalized on WTI)")
print(f"  VECM / Johansen : {np.round(beta_joh, 4)}")
print(f"  State space     : {np.round(beta_cts, 4)}")

print("\nSpread mean reversion")
print(f"  direct AR(1) on the observed spread : rho = {hl_direct['rho']:.4f}, "
      f"half-life = {hl_direct['half_life_weeks']:.2f} weeks")
print(f"  state space (latent component)      : phi = {summary['spread_ar_coefficient']:.4f}, "
      f"half-life = {summary['spread_half_life_weeks']:.2f} weeks")

The two half-lives are the interesting comparison. An AR(1) fitted to the
*observed* spread is attenuated by measurement noise: classical
errors-in-variables biases the autoregressive coefficient towards zero, so the
observed spread looks like it mean-reverts faster than the underlying
dislocation actually does. The state-space estimate separates $\nu_t$ from
$\varepsilon_t$ and is the one to quote — the v0.1 figure of 5.2 weeks should be
read as a lower bound.

## 4. Filtered states

In [ ]:
fs = filtered_states(cts, logp.index)
obs_spread = (logp["WTI"] - logp["BRENT"]).reindex(fs.index)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(fs.index, fs["trend_filtered"], lw=1.2, label="filtered common trend")
axes[0].plot(logp.index, logp["WTI"], lw=0.7, alpha=0.6, label="log WTI")
axes[0].plot(logp.index, logp["BRENT"], lw=0.7, alpha=0.6, label="log Brent")
axes[0].set_title("Common stochastic trend"); axes[0].legend()

axes[1].plot(obs_spread.index, obs_spread, lw=0.8, alpha=0.5, label="observed log spread")
axes[1].plot(fs.index, fs["spread_filtered"], lw=1.3, label="filtered")
axes[1].plot(fs.index, fs["spread_smoothed"], lw=1.0, ls="--", label="smoothed")
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_title("Relative component: observed vs filtered"); axes[1].legend()

noise = obs_spread.values - fs["spread_filtered"].values
axes[2].plot(fs.index, noise, lw=0.7)
axes[2].axhline(0, color="k", lw=0.8)
axes[2].set_title("Observed minus filtered: measurement noise and roll artefacts")

plt.tight_layout(); plt.show()

print(f"sd(observed spread)  {obs_spread.std():.4f}")
print(f"sd(filtered spread)  {fs['spread_filtered'].std():.4f}")
print(f"sd(noise)            {np.std(noise):.4f}")
print(f"signal-to-noise      {summary['signal_to_noise_spread']:.2f}")

The bottom panel is worth reading against `roll_gap_diagnostics()` from
notebook 02: the largest measurement-noise spikes should line up with contract
rolls and with the March-April 2020 dislocation. That is the concrete cost of
using concatenated front-month proxies, and it is exactly what the filter
removes before the scenario engine sees the spread.

## 5. Variance decomposition

In [ ]:
phi = summary["spread_ar_coefficient"]
var_spread = summary["spread_uncond_sd"] ** 2
var_trend_1w = cts.params["sigma_eta"] ** 2

rows = []
for h in [1, 4, 12, 20, 52]:
    v_trend = h * var_trend_1w
    v_spread = var_spread * (1 - phi ** (2 * h))
    rows.append({"horizon_weeks": h,
                 "var_common_trend": v_trend,
                 "var_relative": v_spread,
                 "share_common_%": 100 * v_trend / (v_trend + v_spread)})
display(pd.DataFrame(rows).round(6))

This is the risk-decomposition statement the project has been implying all
along, now quantified: nearly all multi-week WTI variance is the common oil
trend, and the relative component saturates. A hedged WTI-Brent book is exposed
to a bounded risk that stops growing with the horizon, while an outright
position is exposed to a variance that grows linearly. That is why the ablation
finds forecastable structure in the spread and none in the levels.

## 6. Scenario generation from the filtered state

In [ ]:
H = 20
N = 10_000
u = common_uniforms(H, N, seed=2026)

paths = {}
for name, model in [("CTS", cts), ("VECM", vecm)]:
    pool, prob, diag = prepare_residual_pool(model.residuals, mode="time", half_life=52)
    paths[name] = simulate(model, pool, prob, horizon=H, n_sim=N, u=u)
    print(f"{name:5s} pool {tuple(pool.shape)}  ESS {diag.ess:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
h = np.arange(H + 1)
for ax, name in zip(axes, ["CTS", "VECM"]):
    sim = paths[name][:, :, 0] - paths[name][:, :, 1]
    q = np.quantile(sim, [0.05, 0.25, 0.5, 0.75, 0.95], axis=1)
    ax.fill_between(h, q[0], q[4], alpha=0.15)
    ax.fill_between(h, q[1], q[3], alpha=0.3)
    ax.plot(h, q[2], lw=1.5)
    ax.axhline(0, color="k", lw=0.8, ls=":")
    ax.set_title(f"{name}: WTI-Brent log spread scenarios")
    ax.set_xlabel("weeks ahead")
plt.tight_layout(); plt.show()

## 7. Where this sits in the ablation

`CTS_equal`, `CTS_time` and `CTS_macro` are part of the standard model grid in
`validation.py`, so the state space is judged by the same rules as everything
else: it survives only if it beats the matching `VECM_*` with a Diebold-Mariano
p-value below 0.05 and a bootstrap interval excluding zero.
`scripts/run_validation.py` writes that head-to-head to
`significance_cts_vs_vecm.csv` and to the "State space vs VECM" section of
`RESULTS.md`.

Two honest caveats:

1. The measurement-error interpretation assumes the noise is genuinely
   observational. Some of what the filter removes is real, tradeable price
   variation at the front of the curve. With contract-level data this becomes
   testable rather than assumed.
2. The Gaussian likelihood is used for estimation while the innovations are
   bootstrapped empirically for simulation. That is a deliberate quasi-ML
   choice — parameters from a Gaussian likelihood are consistent under
   non-normality, and the tails come from the data — but it should be stated
   rather than glossed over.